# FeO aerosols in rock-vapour atmospheres

How does the FeO content of a molten silicate mantle affect the rock-vapour atmosphere it outgasses, and when does wüstite (FeO) condense into aerosols?

The FeO content of a silicate mantle is set largely by its redox state. A more reduced mantle partitions more iron into metal (the core), leaving less FeO in the silicate. We treat bulk FeO (wt%) as the redox control parameter and, for each composition:

1. **LavAtmos** (Van Buchem et al. 2023, [arXiv:2210.10463](https://arxiv.org/abs/2210.10463)) computes the gas-melt equilibrium above the melt at the surface temperature. It gives the partial pressures of the vaporised species and solves self-consistently for fO$_2$.
2. The surface partial pressures give the total surface pressure and the (well-mixed) volume mixing ratios that are passed to **AGNI** through `juliacall`.
3. AGNI solves for radiative-convective equilibrium with the `Volenfell` spectral file in three stages:
    * `base`: no condensation, used as a reference.
    * `cond`: FeO can condense (rainout and cold-trapping), but aerosol opacity is off.
    * `aer`: as `cond`, plus Mie-scattering wüstite aerosols whose mass is set by the FeO condensate yield.

   Each stage starts from the solution of the previous one. This continuation is needed for robust convergence. When FeO condensation is switched on from a cold initial guess, FeO is cold-trapped out of the whole column and the solver collapses to the temperature floor.

Comparing `cond` with `aer` isolates the radiative effect of the aerosol. Comparing `base` with `cond` isolates the effect of removing FeO from the gas phase.

Latent heat release is excluded throughout (`latent=false`). The limitations of this setup are listed at the end of the notebook.

**Environment.** Run this with a Python kernel that has `thermoengine` (MELTS), `juliacall`, `calliope`, `netCDF4`, `pandas` and `matplotlib` installed (e.g. the `prt` conda env). `RAD_DIR` must point to a compiled SOCRATES, and a compiled FastChem binary must be available.

In [ ]:
import os, sys, time, logging, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import netCDF4 as nc
from calliope.oxygen_fugacity import OxygenFugacity

## Configuration

All of the parameters are set here.

* The planet is a generic Earth-sized lava world with the surface held at the melt temperature (`sol_type=1`).
* The instellation is chosen so that the absorbed stellar flux of a transparent, zero-albedo atmosphere balances $\sigma T_\mathrm{s}^4$. That is, $F = \sigma T_\mathrm{s}^4 / (s_0 \cos\theta)$, with $s_0$ and $\theta$ the AGNI default flux scale factor and zenith angle.

In [ ]:
# Paths
AGNI_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))   # notebook lives in misc/science/
LAVA_DIR = os.path.join(AGNI_DIR, "LavAtmos")
FC_DIR   = os.path.join(AGNI_DIR, "fastchem")
OUT_DIR  = os.path.join(AGNI_DIR, "out", "feo_aerosols")
os.makedirs(OUT_DIR, exist_ok=True)
assert "RAD_DIR" in os.environ, "RAD_DIR must point to a compiled SOCRATES installation"

SPFILE   = os.path.join(AGNI_DIR, "res", "spectral_files", "Volenfell", "Volenfell.sf")
STAR     = os.path.join(AGNI_DIR, "res", "stellar_spectra", "sun.txt")
BASE_MELT = os.path.join(LAVA_DIR, "input", "lava_compositions", "BSE_palm.csv")   # BSE, FeO = 8.1 wt%

# Melt and redox sweep
T_MELT  = 2500.0                                    # surface = melt temperature [K]
FEO_WT  = [0.5, 2.0, 5.0, 8.1, 12.0, 18.0, 25.0]    # bulk FeO in the silicate melt [wt%]

# Planet and star
SIGMA   = 5.670374419e-8
RADIUS  = 6.371e6       # [m]
GRAVITY = 9.81          # [m s-2]
S0_FACT = 0.6652        # AGNI default flux scale factor
ZENITH  = 60.0          # AGNI default zenith angle [deg]
ALBEDO_B = 0.0          # no pseudo-albedo; reflection is computed by the RT
INSTELLATION = SIGMA * T_MELT**4 / (S0_FACT * np.cos(np.radians(ZENITH)))   # [W m-2]

# AGNI grid and solver
NLEV     = 50
P_TOP    = 1e-8         # [bar]
MAX_STEPS = 200
VMR_FLOOR = 1e-10       # drop gases below this VMR before passing them to AGNI

# FeO aerosol: wüstite optical constants (res/refractive/FeO.txt), log-normal size distribution
AEROSOL = dict(method="mie", species="FeO", nk_file="FeO", r_eff=0.5e-6, sigma_g=1.65)

print(f"Instellation = {INSTELLATION:.3e} W m-2  ({INSTELLATION/1361.0:.0f} x solar constant)")

## LavAtmos setup

LavAtmos needs a `paths` object, which it does not define itself. The class below follows the `paths_importer` in PROTEUS (`src/proteus/outgas/lavatmos.py`), with the same attribute names. `fastchem3_output` must end in `/` because LavAtmos concatenates file names onto it.

In [ ]:
os.environ["LAVA_DIR"] = LAVA_DIR
os.environ["FC_DIR"]   = FC_DIR
if LAVA_DIR not in sys.path:
    sys.path.insert(0, LAVA_DIR)
import lavatmos3

class LavaPaths:
    def __init__(self, out):
        self.lavatmos_dir  = LAVA_DIR + "/"
        self.input_dir     = os.path.join(LAVA_DIR, "input") + "/"
        self.fastchem3_dir = FC_DIR + "/"
        self.fastchem3_config_template  = os.path.join(self.input_dir, "fastchem3", "config_template.input")
        self.element_abundance_template = os.path.join(self.input_dir, "fastchem3", "element_abundances",
                                                       "element_abundances_template2.dat")
        self.species_data_file      = os.path.join(FC_DIR, "input", "logK", "logK.dat")
        self.species_data_file_cond = os.path.join(FC_DIR, "input", "logK", "logK_condensates.dat")
        os.makedirs(os.path.join(out, "element_abundances"), exist_ok=True)
        os.makedirs(os.path.join(out, "fastchem"), exist_ok=True)
        self.element_abundance_output = os.path.join(out, "element_abundances", "element_abundances_output.dat")
        self.fastchem3_output = os.path.join(out, "fastchem", "")
        self.output_dir       = self.fastchem3_output
        self.janafdata        = os.path.join(LAVA_DIR, "data")

@contextlib.contextmanager
def in_dir(path):
    # LavAtmos resolves some data paths relative to the working directory
    old = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)

## Melt compositions

We start from the BSE composition and set FeO to each target value, rescaling the other oxides so that their relative proportions and the total are unchanged. None of the compositions contain Fe$_2$O$_3$, and all iron is treated as ferrous.

For reference, we estimate the redox state that each FeO content would imply if the silicate were in equilibrium with a pure iron core, using the reaction $\mathrm{Fe} + \tfrac{1}{2}\mathrm{O_2} = \mathrm{FeO}$:

$$\Delta\mathrm{IW} \approx 2\log_{10}\!\left(\frac{x_\mathrm{FeO}^\mathrm{sil}\,\gamma_\mathrm{FeO}}{x_\mathrm{Fe}^\mathrm{met}\,\gamma_\mathrm{Fe}}\right) \approx 2\log_{10} x_\mathrm{FeO}^\mathrm{sil}$$

This takes ideal mixing ($\gamma = 1$) and a pure iron metal ($x_\mathrm{Fe}^\mathrm{met}=1$), with $x_\mathrm{FeO}$ as the oxide mole fraction. **It is an order-of-magnitude guide only**; non-ideal $\gamma_\mathrm{FeO}$ and alloying elements in the metal would shift it.

In [ ]:
OXIDE_MM = {"SiO2": 60.084, "MgO": 40.304, "Al2O3": 101.961, "TiO2": 79.866, "FeO": 71.844,
            "CaO": 56.077, "Na2O": 61.979, "K2O": 94.196, "Fe2O3": 159.688}   # [g mol-1]

base_melt = {str(k): float(v) for k, v in pd.read_csv(BASE_MELT, header=None).values}
base_total = sum(base_melt.values())

def melt_with_feo(feo_wt):
    # Set FeO to feo_wt and rescale the others so that the total is unchanged
    others = {k: v for k, v in base_melt.items() if k != "FeO"}
    scale = (base_total - feo_wt) / sum(others.values())
    comp = {k: v * scale for k, v in others.items()}
    comp["FeO"] = feo_wt
    return comp

def x_oxide(comp, oxide):
    mol = {k: v / OXIDE_MM[k] for k, v in comp.items()}
    return mol[oxide] / sum(mol.values())

melts = {w: melt_with_feo(w) for w in FEO_WT}
for w, c in melts.items():
    assert abs(sum(c.values()) - base_total) < 1e-9, "composition total not conserved"
    assert all(v >= 0 for v in c.values()), "negative oxide abundance"

melt_df = pd.DataFrame(melts).T
melt_df.index.name = "FeO_target_wt%"
melt_df["x_FeO"] = [x_oxide(melts[w], "FeO") for w in FEO_WT]
melt_df["dIW_core_ideal"] = 2 * np.log10(melt_df["x_FeO"])
melt_df.round(4)

## Vaporisation with LavAtmos

For each melt, `vaporise` returns FastChem partial pressures [bar] for the gas above the melt. The atmosphere is taken to be free of volatiles: C, H, N, S and P are included only as traces. LavAtmos requires `O` to be present in `volatile_comp`; without it the inner oxygen-abundance solve is skipped.

**Bracketing the fO$_2$ solve.** By default LavAtmos searches for fO$_2$ within a bracket taken from a hard-coded T–fO$_2$ table. At 2500 K that bracket is $10^{-15}$–$10^{-5}$ bar. For BSE, the mass-balance residual stays positive across that whole range. The solver then falls back to minimising the absolute residual and simply returns the upper bound, which is not a solution. The real root lies near $10^{-2.6}$ bar, consistent with congruent vaporisation (e.g. $\mathrm{Na_2O \rightarrow 2Na + \tfrac12 O_2}$ gives $p_\mathrm{O_2}\sim p_\mathrm{Na}/4$).

We therefore pass an explicit initial guess (`fO2_tries_from_last=True`), which brackets $\pm 2$ decades around it. Each case warm-starts from the previous one. We also capture LavAtmos's log so that any case where no root was bracketed is flagged, rather than silently accepted.

In [ ]:
class _LogCapture(logging.Handler):
    def __init__(self):
        super().__init__(level=logging.DEBUG)
        self.records = []
    def emit(self, record):
        self.records.append(record.getMessage())

VOLATILES = {"C": 1e-20, "H": 1e-9, "N": 1e-20, "S": 1e-20, "P": 1e-20, "O": 1e-20}   # trace only
P_VOLATILE = 1e-8     # [bar]
iw_buffer = OxygenFugacity()   # CALLIOPE default IW parametrisation, as used by PROTEUS

def vaporise(feo_wt, fO2_guess):
    cache = os.path.join(OUT_DIR, f"lavatmos_feo{feo_wt:05.2f}.csv")
    if os.path.exists(cache):
        return pd.read_csv(cache, index_col=0).iloc[0], {"cached": True}
    work = os.path.join(OUT_DIR, "lavatmos_work", f"feo{feo_wt:05.2f}")
    system = lavatmos3.melt_vapor_system(LavaPaths(work))
    cap = _LogCapture()
    lg = logging.getLogger("fwl.lavatmos3")
    lg.setLevel(logging.DEBUG); lg.addHandler(cap)
    try:
        with in_dir(LAVA_DIR):
            res = system.vaporise(T_MELT, P_VOLATILE, melts[feo_wt], VOLATILES, verbose=False,
                                  fO2_initial_guess=fO2_guess, fO2_tries_from_last=True)
    finally:
        lg.removeHandler(cap)
    info = {"cached": False,
            "outer_fallback": any("outer fO2 solve" in m and "do not bracket" in m for m in cap.records),
            "inner_fallbacks": sum("inner O_abun solve" in m and "do not bracket" in m for m in cap.records)}
    row = res.iloc[0]
    row.to_csv(cache)
    pd.Series(info).to_csv(cache.replace(".csv", "_info.csv"))
    return row, info

lava = {}
fO2_guess = 1e-3       # [bar], see the note above
for w in FEO_WT:
    t0 = time.time()
    row, info = vaporise(w, fO2_guess)
    if not info.get("cached"):
        print(f"FeO = {w:5.2f} wt%: {time.time()-t0:5.1f} s, outer fallback = {info['outer_fallback']}, "
              f"inner fallbacks = {info['inner_fallbacks']}")
    else:
        info_file = os.path.join(OUT_DIR, f"lavatmos_feo{w:05.2f}_info.csv")
        info = pd.read_csv(info_file, index_col=0).iloc[:, 0].to_dict()
        print(f"FeO = {w:5.2f} wt%: loaded from cache, outer fallback = {info['outer_fallback']}")
    lava[w] = (row, info)
    fO2_guess = float(row["O2"])     # warm start the next case

### From FastChem species to AGNI gases

FastChem labels species in Hill-like notation (`Fe1O1`, `O1Si1`, and so on). We map them to AGNI gas names, drop ions and electrons, and discard any gas with VMR below `VMR_FLOOR`. Every dropped species is reported. The total surface pressure is the sum of the retained neutral partial pressures.

In [ ]:
FC_TO_AGNI = {"Na": "Na", "K": "K", "Fe": "Fe", "Mg": "Mg", "Si": "Si", "Ca": "Ca", "Al": "Al", "Ti": "Ti",
              "O": "O", "O2": "O2", "Fe1O1": "FeO", "O1Si1": "SiO", "O2Si1": "SiO2", "Mg1O1": "MgO",
              "O1Ti1": "TiO", "O2Ti1": "TiO2", "Ca1O1": "CaO", "Al1O1": "AlO", "Na1O1": "NaO",
              "Mg2": "Mg2", "Na2": "Na2"}

def to_agni(row):
    neutral = row[[c for c in row.index if not c.endswith(("+", "-"))]]
    neutral = neutral[neutral > 0]
    p_tot_all = neutral.sum()
    pp = {FC_TO_AGNI[k]: float(v) for k, v in neutral.items() if k in FC_TO_AGNI}
    unmapped = neutral[[k for k in neutral.index if k not in FC_TO_AGNI]]
    p_mapped = sum(pp.values())
    vmr = {g: p / p_mapped for g, p in pp.items() if p / p_mapped >= VMR_FLOOR}
    norm = sum(vmr.values())
    vmr = {g: v / norm for g, v in vmr.items()}
    lost = 1.0 - (p_mapped * norm) / p_tot_all
    return vmr, p_mapped * norm, lost, unmapped

surf = []
agni_in = {}
for w in FEO_WT:
    row, info = lava[w]
    vmr, psurf, lost, unmapped = to_agni(row)
    assert abs(sum(vmr.values()) - 1.0) < 1e-12
    agni_in[w] = (vmr, psurf)
    log_fO2 = np.log10(float(row["O2"]))
    surf.append(dict(FeO_wt=w, p_surf_bar=psurf, frac_pressure_dropped=lost,
                     log_fO2=log_fO2, dIW_atm=log_fO2 - iw_buffer(T_MELT),
                     dIW_core_ideal=melt_df.loc[w, "dIW_core_ideal"],
                     p_Fe=float(row.get("Fe", 0)),
                     p_FeO=float(row.get("Fe1O1", 0)),
                     lavatmos_outer_fallback=bool(info["outer_fallback"])))
    if len(unmapped):
        print(f"FeO={w:5.2f}: unmapped neutrals (bar): " +
              ", ".join(f"{k}={v:.1e}" for k, v in unmapped.sort_values(ascending=False).head(5).items()))
surf_df = pd.DataFrame(surf).set_index("FeO_wt")
surf_df

Now we look at how the surface partial pressures change with melt FeO. The top axis shows the atmospheric ΔIW that LavAtmos returns.

Colours mark species identity, and each line is also labelled directly, so the plot can be read without relying on colour. The eight most abundant species (ranked across all cases) are plotted.

In [ ]:
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
FEO_RAMP = ["#86b6ef", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#184f95", "#0d366b"]   # ordinal blue
feo_col = {w: FEO_RAMP[i] for i, w in enumerate(FEO_WT)}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e4e3dd", "grid.linewidth": 0.6, "lines.linewidth": 2})

pp_tab = pd.DataFrame({w: {g: v * agni_in[w][1] for g, v in agni_in[w][0].items()} for w in FEO_WT}).T.fillna(0)
top = pp_tab.max().sort_values(ascending=False).index[:8]

fig, ax = plt.subplots(figsize=(7, 4.5))
for i, g in enumerate(top):
    ax.plot(pp_tab.index, pp_tab[g], color=CAT[i], marker="o", ms=4)
    ax.annotate(g, (pp_tab.index[-1], pp_tab[g].iloc[-1]), xytext=(4, 0), textcoords="offset points",
                va="center", fontsize=9, color="#3d3d3a")
ax.plot(surf_df.index, surf_df.p_surf_bar, color="#3d3d3a", ls=":", lw=1.5)
ax.annotate("total", (surf_df.index[-1], surf_df.p_surf_bar.iloc[-1]), xytext=(4, 0),
            textcoords="offset points", va="center", fontsize=9, color="#3d3d3a")
ax.set_yscale("log"); ax.set_xlabel("Melt FeO [wt%]"); ax.set_ylabel("Surface partial pressure [bar]")
ax.set_ylim(bottom=max(pp_tab[top].values[pp_tab[top].values > 0].min() / 3, 1e-9))
sec = ax.secondary_xaxis("top")
sec.set_xticks(surf_df.index[[0, 3, -1]])
sec.set_xticklabels([f"{v:+.1f}" for v in surf_df.dIW_atm.iloc[[0, 3, -1]]])
sec.set_xlabel("Atmospheric ΔIW (LavAtmos)")
ax.set_title(f"Rock vapour above the melt at {T_MELT:.0f} K", loc="left")
fig.tight_layout(); plt.show()

## AGNI via `juliacall`

This follows the calling pattern in PROTEUS (`src/proteus/atmos_clim/agni.py`). In Python, the `!` of Julia's mutating functions is written `_b`.

The aerosol table is built directly in Julia as a `Dict{String,Any}`. Validating it (`parse_aerosol_entry`) involves string and number comparisons on the entries, which is safest with native Julia types.

In [ ]:
from juliacall import Main as jl
jl.seval("using Pkg")
jl.Pkg.activate(AGNI_DIR)
jl.seval("import AGNI")
jl.AGNI.setup_logging(os.path.join(OUT_DIR, "agni.log"), 1)

AER_JL = jl.seval('Dict{String,Any}("FeO" => Dict{String,Any}(' + ", ".join(
    f'"{k}" => {repr(v) if isinstance(v, float) else chr(34) + v + chr(34)}' for k, v in AEROSOL.items()) + "))")
print(AER_JL)

In [ ]:
def run_agni(tag, vmr, psurf, condense, aerosol, init_nc=None):
    # Solve one RCE case with fixed T_surf = T_MELT, write a NetCDF file, and return diagnostics as numpy arrays
    atmos = jl.AGNI.atmosphere.Atmos_t()
    kw = dict(flag_aerosol=True, aerosol_species=AER_JL) if aerosol else dict(flag_aerosol=False)
    ok = jl.AGNI.atmosphere.setup_b(atmos, AGNI_DIR, OUT_DIR, SPFILE,
                                    INSTELLATION, S0_FACT, ALBEDO_B, ZENITH, T_MELT,
                                    GRAVITY, RADIUS, NLEV, psurf, P_TOP,
                                    jl.convert(jl.Dict, vmr), "",
                                    condensates=jl.convert(jl.Array[jl.String, 1], ["FeO"] if condense else []),
                                    flag_rayleigh=True, tmp_floor=50.0, check_integrity=False, **kw)
    assert bool(ok), f"{tag}: setup! failed"
    assert bool(jl.AGNI.atmosphere.allocate_b(atmos, STAR)), f"{tag}: allocate! failed"

    if init_nc is None:
        jl.AGNI.setpt.isothermal_b(atmos, 0.8 * T_MELT)
    else:
        jl.AGNI.setpt.fromncdf_b(atmos, init_nc)

    t0 = time.time()
    try:
        conv = bool(jl.AGNI.solver.solve_energy_b(atmos, sol_type=1, chem=False, rainout=condense,
                                                  latent=False, convect=True, sens_heat=True,
                                                  conduct=False, oceans=False, max_steps=MAX_STEPS,
                                                  easy_start=False, save_frames=False, modplot=0))
    except Exception as e:     # Julia-side exception, treat as non-converged
        print(f"    {tag}: solver raised {type(e).__name__}: {str(e)[:120]}")
        conv = False

    path = os.path.join(OUT_DIR, f"{tag}.nc")
    jl.AGNI.save.write_ncdf(atmos, path)

    gas = lambda g: np.array(atmos.gas_vmr[g]) if g in atmos.gas_names else np.zeros(NLEV)
    out = dict(tag=tag, converged=conv, runtime=time.time() - t0, nc=path,
               p=np.array(atmos.p), pl=np.array(atmos.pl), tmp=np.array(atmos.tmp), tmpl=np.array(atmos.tmpl),
               flux_n=np.array(atmos.flux_n), flux_tot=np.array(atmos.flux_tot),
               flux_u_lw=np.array(atmos.flux_u_lw), flux_u_sw=np.array(atmos.flux_u_sw),
               flux_d_sw=np.array(atmos.flux_d_sw),
               band_u_lw=np.array(atmos.band_u_lw), band_u_sw=np.array(atmos.band_u_sw),
               bands_min=np.array(atmos.bands_min), bands_max=np.array(atmos.bands_max),
               x_FeO=gas("FeO"), x_Fe=gas("Fe"),
               layer_mass=np.array(atmos.layer_σ),
               cond_yield=np.array(atmos.cond_yield["FeO"]) if condense else np.zeros(NLEV),
               aer_mmr=np.array(atmos.aerosol_arr_l["feo"]) if aerosol else np.zeros(NLEV))
    jl.AGNI.atmosphere.deallocate_b(atmos)
    print(f"    {tag:22s} converged={conv!s:5s} {out['runtime']:6.1f} s   "
          f"T=[{out['tmp'].min():.0f}, {out['tmp'].max():.0f}] K   "
          f"FeO yield={out['cond_yield'].sum():.2e} kg m-2")
    return out

## Sweep

For each FeO content we run the `base` → `cond` → `aer` sequence. Runs that do not converge are kept, but they are flagged in every figure (hollow markers) and in the checks table.

In [ ]:
runs = {}
prev_base = None
for w in FEO_WT:
    vmr, psurf = agni_in[w]
    print(f"FeO = {w:5.2f} wt%   p_surf = {psurf:.3e} bar")
    stem = f"feo{w:05.2f}"
    r_base = run_agni(stem + "_base", vmr, psurf, condense=False, aerosol=False, init_nc=prev_base)
    r_cond = run_agni(stem + "_cond", vmr, psurf, condense=True,  aerosol=False, init_nc=r_base["nc"])
    r_aer  = run_agni(stem + "_aer",  vmr, psurf, condense=True,  aerosol=True,  init_nc=r_cond["nc"])
    runs[w] = dict(base=r_base, cond=r_cond, aer=r_aer)
    if r_base["converged"]:
        prev_base = r_base["nc"]     # warm start the next composition

## Temperature structure and FeO saturation

The FeO saturation vapour pressure comes from the same thermodynamic file that AGNI uses (`res/thermodynamics/FeO.nc`, `sat_P` in log$_{10}$ Pa). The saturation ratio $S = p_\mathrm{FeO}/p_\mathrm{sat}(T)$ shows how close each layer is to condensing. Wherever FeO condenses, the `cond` and `aer` profiles hold $S \le 1$.

In [ ]:
with nc.Dataset(os.path.join(AGNI_DIR, "res", "thermodynamics", "FeO.nc")) as ds:
    _satT, _satP = np.array(ds["sat_T"][:]), np.array(ds["sat_P"][:])
psat_FeO = lambda T: 10.0 ** np.interp(T, _satT, _satP)     # [Pa]

def sat_ratio(r):
    return r["x_FeO"] * r["p"] / psat_FeO(r["tmp"])

fig, axs = plt.subplots(1, 3, figsize=(12, 5), sharey=True)
for w in FEO_WT:
    c = feo_col[w]
    for k, ls in [("base", ":"), ("cond", "--"), ("aer", "-")]:
        r = runs[w][k]
        alpha = 1.0 if r["converged"] else 0.35
        axs[0].plot(r["tmp"], r["p"] / 1e5, color=c, ls=ls, alpha=alpha)
        if k != "base":
            axs[1].plot(sat_ratio(r), r["p"] / 1e5, color=c, ls=ls, alpha=alpha)
    r = runs[w]["aer"]
    axs[2].plot(np.maximum(r["aer_mmr"], 1e-30), r["p"] / 1e5, color=c, alpha=1.0 if r["converged"] else 0.35,
                label=f"{w:g} wt%")
axs[0].set_xlabel("Temperature [K]"); axs[0].set_ylabel("Pressure [bar]")
axs[1].set_xlabel(r"FeO saturation ratio $S$"); axs[1].set_xscale("log"); axs[1].axvline(1, color="#3d3d3a", lw=1)
axs[2].set_xlabel("FeO aerosol MMR [kg/kg]"); axs[2].set_xscale("log"); axs[2].set_xlim(left=1e-12)
axs[0].set_yscale("log"); axs[0].invert_yaxis()
axs[2].legend(title="Melt FeO", fontsize=8, frameon=False)
axs[0].plot([], [], color="#3d3d3a", ls=":", label="base (no condensation)")
axs[0].plot([], [], color="#3d3d3a", ls="--", label="cond (aerosol opacity off)")
axs[0].plot([], [], color="#3d3d3a", ls="-", label="aer (aerosol opacity on)")
axs[0].legend(fontsize=8, frameon=False, loc="lower left")
axs[0].set_title("T(p)", loc="left"); axs[1].set_title("Saturation", loc="left"); axs[2].set_title("Aerosol", loc="left")
fig.tight_layout(); plt.show()

## Summary against melt FeO

* The **OLR** is the upward longwave flux at the top of the atmosphere.
* The **Bond albedo** is the upward divided by the downward shortwave flux at the top.
* The **aerosol column** is $\sum_i \mathrm{MMR}_i\,\sigma_i$, where $\sigma_i$ is the layer column mass.
* The **stratospheric ΔT** is the mean temperature difference, `aer` minus `cond`, above 1 Pa.

Hollow markers indicate that the run did not converge.

In [ ]:
def summary(r):
    return dict(OLR=r["flux_u_lw"][0], albedo=r["flux_u_sw"][0] / r["flux_d_sw"][0],
                net_TOA=r["flux_n"][0], aer_col=float(np.sum(r["aer_mmr"] * r["layer_mass"])),
                yield_col=float(np.sum(r["cond_yield"])), T_min=r["tmp"].min(), converged=r["converged"])

rows = []
for w in FEO_WT:
    for k in ("base", "cond", "aer"):
        rows.append(dict(FeO_wt=w, stage=k, **summary(runs[w][k])))
summ = pd.DataFrame(rows).set_index(["FeO_wt", "stage"])
dT = {w: np.mean((runs[w]["aer"]["tmp"] - runs[w]["cond"]["tmp"])[runs[w]["aer"]["p"] < 1.0]) for w in FEO_WT}

fig, axs = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
panels = [("OLR", "OLR [W m$^{-2}$]"), ("albedo", "Bond albedo"), ("aer_col", "Aerosol column [kg m$^{-2}$]")]
for ax, (key, lab) in zip(axs.flat[:3], panels):
    for k, ls, col in [("base", ":", "#898781"), ("cond", "--", "#1c5cab"), ("aer", "-", "#eb6834")]:
        sub = summ.xs(k, level="stage")
        ax.plot(sub.index, sub[key], ls=ls, color=col, label=k)
        conv = sub["converged"].values.astype(bool)
        ax.plot(sub.index[conv], sub[key][conv], "o", ms=5, color=col)
        ax.plot(sub.index[~conv], sub[key][~conv], "o", ms=7, mfc="none", color=col)
    ax.set_ylabel(lab)
axs[0, 0].legend(frameon=False, fontsize=8)
ax = axs[1, 1]
ax.plot(FEO_WT, [dT[w] for w in FEO_WT], color="#eb6834", marker="o", ms=5)
ax.axhline(0, color="#3d3d3a", lw=1)
ax.set_ylabel(r"$\Delta T$ aer$-$cond, p < 1 Pa [K]")
for ax in axs[1]:
    ax.set_xlabel("Melt FeO [wt%]")
fig.suptitle("Radiative effect of FeO removal and FeO aerosol", x=0.01, ha="left")
fig.tight_layout(); plt.show()
summ.round(4)

### Emission spectrum

Here is the band-resolved outgoing flux at the top of the atmosphere (longwave + reflected shortwave) for the lowest, BSE and highest FeO cases, with aerosol opacity on (solid) and off (dashed). The flux is divided by the band width, and the bands are the 64 `Volenfell` bands.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for w in (FEO_WT[0], 8.1, FEO_WT[-1]):
    for k, ls in [("cond", "--"), ("aer", "-")]:
        r = runs[w][k]
        wl_um = 0.5 * (r["bands_min"] + r["bands_max"]) * 1e6
        dwl_um = (r["bands_max"] - r["bands_min"]) * 1e6
        f = (r["band_u_lw"][0, :] + r["band_u_sw"][0, :]) / dwl_um
        ax.step(wl_um, f, where="mid", color=feo_col[w], ls=ls, lw=1.5,
                label=f"{w:g} wt% ({k})")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(0.2, 50)
ax.set_xlabel("Wavelength [µm]"); ax.set_ylabel(r"Upward flux at TOA [W m$^{-2}$ µm$^{-1}$]")
ax.legend(frameon=False, fontsize=8, ncol=3)
fig.tight_layout(); plt.show()

## Physical checks

These are necessary conditions, not a validation of the physics.

* The temperature is positive and finite.
* VMRs sum to 1 at the surface.
* Fluxes are finite.
* For converged runs, the net flux is uniform with height to within 1% (energy conservation for `sol_type=1`, where the uniform net flux is the flux the surface must supply).
* Aerosol mass is zero when aerosols are off and non-negative otherwise.
* FeO is not supersaturated ($S \le 1+10^{-6}$) where condensation is on.

In [ ]:
chk = []
for w in FEO_WT:
    vmr, _ = agni_in[w]
    for k in ("base", "cond", "aer"):
        r = runs[w][k]
        fn = r["flux_n"]
        spread = (fn.max() - fn.min()) / max(abs(fn).max(), 1e-30)
        S = sat_ratio(r)
        chk.append(dict(FeO_wt=w, stage=k, converged=r["converged"],
                        T_positive=bool(np.all(np.isfinite(r["tmpl"])) and np.all(r["tmpl"] > 0)),
                        vmr_sum_1=abs(sum(vmr.values()) - 1) < 1e-12,
                        flux_finite=bool(np.all(np.isfinite(r["flux_tot"]))),
                        flux_n_spread=spread,
                        energy_ok=(spread < 1e-2) if r["converged"] else np.nan,
                        aer_mass_ok=bool(np.all(r["aer_mmr"] >= 0)) and (k == "aer" or np.all(r["aer_mmr"] == 0)),
                        max_S=float(S.max()),
                        no_supersat=(S.max() <= 1 + 1e-6) if k != "base" else np.nan))
chk_df = pd.DataFrame(chk).set_index(["FeO_wt", "stage"])
hard = ["T_positive", "vmr_sum_1", "flux_finite", "aer_mass_ok"]
print("Hard checks all pass:", bool(chk_df[hard].all().all()))
print("Converged runs:", int(chk_df.converged.sum()), "/", len(chk_df))
print("Converged runs conserving energy:", chk_df.energy_ok.dropna().astype(bool).all())
print("LavAtmos fO2 solve bracketed a root in all cases:", not surf_df.lavatmos_outer_fallback.any())
chk_df

## Caveats and limitations

* **FeO gas has no opacity in `Volenfell`.** The spectral file includes the absorbers O$_2$, SiO, TiO, MgO, Na, K and Fe, plus the O$_2$–O$_2$ continuum. FeO(g) therefore affects the result only through its mass and condensation. The only radiative carriers of iron are Fe(g) and the wüstite aerosol. Adding FeO line opacity could change the thermal structure, in particular any upper-atmosphere inversion.
* **The aerosol treatment is parametric.** The aerosol mass is `cloud_alpha` × condensate yield per layer (AGNI default `cloud_alpha = 0.01`). The size distribution is fixed (r$_\mathrm{eff}$ = 0.5 µm, σ$_g$ = 1.65), and there is no sedimentation, coagulation or nucleation. The absolute aerosol loading therefore scales directly with this assumption.
* **Latent heat is excluded** (`latent=false`). AGNI has no latent-heat data for FeO in any case.
* **The atmosphere is well mixed.** The surface VMRs from LavAtmos apply to the whole column, and only condensation modifies them; there is no thermochemistry aloft (`chem=false`). At lower temperatures aloft, gas-phase speciation (e.g. Fe + O ⇌ FeO) would shift.
* **Redox is represented through FeO content.** LavAtmos takes no fO$_2$ input; it solves for fO$_2$ from mass balance. None of the melts include Fe$_2$O$_3$. The core-mantle ΔIW is an ideal-mixing estimate. From reading `lavatmos3.py` (not tested here), the Fe mass-balance weight corresponds to Fe$_2$O$_3$ stoichiometry, which may bias fO$_2$ for iron-rich melts.
* **The LavAtmos fO$_2$ bracket needs care.** See the note in the vaporisation section. The flags in `surf_df` and the checks record whether a root was bracketed in each case.
* **The planet is a single 1D column** with a fixed surface temperature. In `sol_type=1` the surface flux needed to hold $T_\mathrm{s}$ is not constrained. A dayside-only column of a tidally locked lava planet would need a different flux scaling.